In [1]:
import pandas as pd 
import numpy as np 

In [9]:
import pyarrow.parquet as pq

table = pq.read_table("../data_logs/logs_export.parquet/logs_export.parquet")
df_raw = table.to_pandas()
print(df_raw.shape)
print(df_raw.head())

(11997652, 1)
                                             raw_log
0  2025-03-20 01:29:24;94.102.61.47;159.84.146.99...
1  2025-03-20 01:29:25;176.111.174.85;159.84.146....
2  2025-03-20 01:29:27;66.249.65.106;159.84.146.9...
3  2025-03-20 01:29:34;89.248.163.75;159.84.146.9...
4  2025-03-20 01:29:38;42.58.163.244;159.84.146.9...


In [10]:
# parsing 
cols = ["datetime", "ip_src", "ip_dst", "protocol", "port_src", "port_dst", "rule_id", "action", "interface_in", "interface_out", "fw"]

df = df_raw["raw_log"].str.strip().str.split(";", expand=True)
df.columns = cols
print(df.shape)
print(df.head())

(11997652, 11)
              datetime          ip_src         ip_dst protocol port_src  \
0  2025-03-20 01:29:24    94.102.61.47  159.84.146.99      TCP    52502   
1  2025-03-20 01:29:25  176.111.174.85  159.84.146.99      TCP    48739   
2  2025-03-20 01:29:27   66.249.65.106  159.84.146.99      TCP    50501   
3  2025-03-20 01:29:34   89.248.163.75  159.84.146.99      TCP    43312   
4  2025-03-20 01:29:38   42.58.163.244  159.84.146.99      TCP     9746   

  port_dst rule_id  action interface_in interface_out fw  
0     3178     999    DENY         eth0                6  
1     2231     999    DENY         eth0                6  
2      443       1  PERMIT         eth0                6  
3     8845     999    DENY         eth0                6  
4       23       7    DENY         eth0                6  


In [14]:
print(df.dtypes)
print(df.head(3))

datetime         object
ip_src           object
ip_dst           object
protocol         object
port_src         object
port_dst         object
rule_id          object
action           object
interface_in     object
interface_out    object
fw               object
dtype: object
              datetime          ip_src         ip_dst protocol port_src  \
0  2025-03-20 01:29:24    94.102.61.47  159.84.146.99      TCP    52502   
1  2025-03-20 01:29:25  176.111.174.85  159.84.146.99      TCP    48739   
2  2025-03-20 01:29:27   66.249.65.106  159.84.146.99      TCP    50501   

  port_dst rule_id  action interface_in interface_out fw  
0     3178     999    DENY         eth0                6  
1     2231     999    DENY         eth0                6  
2      443       1  PERMIT         eth0                6  


In [15]:
df = df.drop(columns=["fw", "interface_out"])
df["datetime"] = pd.to_datetime(df["datetime"])
df["port_src"] = pd.to_numeric(df["port_src"])
df["port_dst"] = pd.to_numeric(df["port_dst"])
df["rule_id"] = pd.to_numeric(df["rule_id"])

print(df.dtypes)
print(df.shape)

datetime        datetime64[ns]
ip_src                  object
ip_dst                  object
protocol                object
port_src               float64
port_dst               float64
rule_id                float64
action                  object
interface_in            object
dtype: object
(11997652, 9)


In [16]:
df.to_parquet("../data_logs/logs_clean.parquet", index=False)

ArrowKeyError: A type extension with name pandas.period already defined